# Arctic Wolf Ticket API — Getting Started Notebook

Python target: **3.14.6**

This notebook is a practical, API-first starter for working with the Arctic Wolf Ticket API. It is written for builders who want copyable Python examples, safe defaults, and a small client wrapper they can extend.

Scope:

- Authenticate with a bearer token.
- List tickets for an organization.
- Filter tickets by status, priority, type, assignee, and timestamps.
- Retrieve a ticket by ID.
- Add a public comment to a ticket.
- Close a ticket with an optional comment.
- Retrieve a pre-signed attachment download URL.
- Handle pagination and API errors.

Out of scope:

- ITSM synchronization design.
- ServiceNow, Jira, Zendesk, or other ticketing platform integration logic.
- Webhook orchestration.
- Production secret management architecture.

## API surface used in this notebook

Base URLs are region-specific. Set the correct host for your organization’s deployment in the environment rather than hard-coding it in this notebook.

Endpoints covered:

| Method | Path | Purpose |
|---|---|---|
| `GET` | `/api/v1/organizations/{organizationUuid}/tickets` | List tickets for an organization. |
| `GET` | `/api/v1/organizations/{organizationUuid}/tickets/{ticketId}` | Retrieve one ticket by ID. |
| `POST` | `/api/v1/organizations/{organizationUuid}/tickets/{ticketId}/comments` | Add a comment to a ticket. |
| `POST` | `/api/v1/organizations/{organizationUuid}/tickets/{ticketId}/close` | Close a ticket. |
| `GET` | `/api/v1/organizations/{organizationUuid}/tickets/{ticketId}/attachments/{attachmentId}` | Get a pre-signed attachment download URL. |

Authentication uses HTTP bearer auth:

```http
Authorization: Bearer <token>
```


## Install dependencies

This notebook only requires `requests`.

## Configuration

Set these values before running API calls.

Recommended local workflow:

1. Store the bearer token in an environment variable.
2. Set the deployment endpoint in the environment (optional - derived from the resolved organization's POD otherwise).

Required environment variables:

- `AW_TICKET_API_TOKEN` or `PAK_TOKEN`

Optional environment variables:

- `AW_ORGANIZATION_UUID` - only needed if this PAK has access to more than one organization and you want to skip the interactive prompt below.
- `REGIONAL_ENDPOINT` (host for your deployment)

The organization UUID is resolved directly from the PAK via the Organizations API rather than typed in by hand - that avoids pasting the wrong value (a URL, or the `customerID` instead of the `id` field) into config, which otherwise only surfaces as a confusing error several calls later.


In [ ]:
import os
from pathlib import Path
from getpass import getpass
import dotenv

from ticket_api_client import ArcticWolfTicketApiClient
from organizations_client import prompt_for_organization_choice

# Load environment variables from .env
dotenv_path = Path.cwd() / ".env"
print(f"Loading environment from: {dotenv_path}")
dotenv.load_dotenv(dotenv_path, override=True)

TOKEN = os.getenv("AW_TICKET_API_TOKEN") or os.getenv("PAK_TOKEN")
if not TOKEN:
    TOKEN = getpass("PAK_TOKEN: ")

REGIONAL_ENDPOINT = (os.getenv("REGIONAL_ENDPOINT") or os.getenv("BASE_URL") or "").rstrip("/") or None

client = ArcticWolfTicketApiClient.from_pak(
    pak_token=TOKEN,
    organization_uuid=os.getenv("AW_ORGANIZATION_UUID") or None,
    base_url=REGIONAL_ENDPOINT,
    on_multiple_organizations=prompt_for_organization_choice,
)

# Kept for the example cells below, which reference ORGANIZATION_UUID directly.
ORGANIZATION_UUID = client.organization_uuid
BASE_URL = client.base_url

print("✅ Configuration resolved from PAK")
print(f"Base URL: {BASE_URL}")
print(f"Organization UUID: {ORGANIZATION_UUID}")


## Core client

The client below keeps the examples short while preserving explicit request behavior.

Design choices:

- Uses a persistent `requests.Session`.
- Applies bearer token auth once at session creation.
- Converts list query parameters into comma-separated strings because the API accepts single values or comma-separated lists for filters such as `status`, `priority`, and `type`.
- Raises a custom exception containing HTTP status, API error code, API error description, and raw response text when available.

In [ ]:
from ticket_api_client import ArcticWolfTicketApiClient, TicketApiError

# The client used in this notebook lives in ticket_api_client.py, not inline here,
# so fixes and improvements apply everywhere it's used instead of drifting between
# two copies of the same logic.


## Create the API client

In [ ]:
# `client` was already constructed in the Setup and Authentication cell above via
# ArcticWolfTicketApiClient.from_pak(...), which also resolved ORGANIZATION_UUID.
print("✅ Client ready")
print(repr(client.base_url))


## Example 1 — List tickets

This retrieves the first page of tickets for the organization.

Pagination defaults:

- `offset`: `0`
- `limit`: `20`

The API limit parameter supports values from `1` to `100`.

In [ ]:
try:
    page = client.list_tickets(
        ORGANIZATION_UUID,
        offset=0,
        limit=20,
        include_comments=False,
    )
    print(f"Returned: {len(page.get('results', []))}")
    print(f"Meta: {page.get('meta')}")
    print(page.get("results", [])[:3])
except TicketApiError as exc:
    print(exc)

## Example 2 — List open customer-action tickets

Ticket status mapping:

- `OPEN`, `NEW`, `HOLD`: With Arctic Wolf
- `PENDING`: With Customer
- `CLOSED`: Closed
- `OPEN`, `NEW`, `HOLD`, `PENDING`: Open

This example returns tickets that are pending customer action.

In [ ]:
try:
    pending_customer_tickets = client.list_tickets(
        ORGANIZATION_UUID,
        status="PENDING",
        limit=20,
        include_comments=False,
    )
    results = pending_customer_tickets.get("results", [])
    print(f"✅ Found {len(results)} pending customer tickets")
    print(results[:5])
except TicketApiError as exc:
    print(exc)

## Example 3 — Filter by priority, type, and creation time

Supported priorities:

- `LOW`
- `NORMAL`
- `HIGH`
- `URGENT`

Supported ticket types:

- `QUESTION`
- `INCIDENT`
- `PROBLEM`
- `TASK`

Timestamp filters use ISO 8601 UTC values such as `2026-03-01T00:00:00Z`.

In [ ]:
try:
    high_priority_incidents = client.list_tickets(
        ORGANIZATION_UUID,
        priority=["HIGH", "URGENT"],
        ticket_type="INCIDENT",
        created_after="2026-01-01T00:00:00Z",
        limit=50,
        include_comments=False,
    )
    # Previously this line's result wasn't assigned, so the print below silently
    # reused the `results` variable left over from Example 2 instead of this
    # query's actual results.
    results = high_priority_incidents.get("results", [])
    print(f"✅ Found {len(results)} incident tickets marked as HIGH or URGENT priority")
    print(results[:5])
except TicketApiError as exc:
    print(exc)

## Example 4 — Fetch all pages

Use this helper when you want all matching tickets instead of one page.

Guardrails:

- Uses API pagination via `offset` and `limit`.
- Sets `limit=100`, the documented maximum for list requests.
- Stops when the API returns fewer records than requested or when `offset + returned >= total`.

In [9]:
def fetch_all_tickets(
    client: ArcticWolfTicketApiClient,
    organization_uuid: str,
    *,
    limit: int = 100,
    max_pages: int = 100,
    **filters: Any,
) -> list[dict[str, Any]]:
    all_results: list[dict[str, Any]] = []
    offset = 0

    for _ in range(max_pages):
        page = client.list_tickets(
            organization_uuid,
            offset=offset,
            limit=limit,
            **filters,
        )
        results = page.get("results", [])
        meta = page.get("meta", {})
        all_results.extend(results)

        total = meta.get("total")
        returned = len(results)
        offset += returned

        if returned < limit:
            break
        if isinstance(total, int) and offset >= total:
            break

    return all_results


try:
    open_tickets = fetch_all_tickets(
        client,
        ORGANIZATION_UUID,
        status=["OPEN", "NEW", "HOLD", "PENDING"],
        include_comments=False,
    )
    print(f"Fetched {len(open_tickets)} open tickets")
except TicketApiError as exc:
    print(exc)

Fetched 34 open tickets


## Example 5 — Convert ticket results into a DataFrame

This is useful for quick inspection, export, or sorting. It is not required for API usage.

In [10]:
import pandas as pd
import json


def tickets_to_dataframe(tickets: list[dict[str, Any]]) -> pd.DataFrame:
    rows = []
    for ticket in tickets:
        assignee = ticket.get("assignee") or {}
        attachments = ticket.get("attachments") or []
        attachment_ids = [att.get("id") for att in attachments if not att.get("deleted")]
        rows.append({
            "id": ticket.get("id"),
            "title": ticket.get("title"),
            "status": ticket.get("status"),
            "priority": ticket.get("priority"),
            "type": ticket.get("type"),
            "createdAt": ticket.get("createdAt"),
            "updatedAt": ticket.get("updatedAt"),
            "commentCount": ticket.get("commentCount"),
            "attachmentCount": ticket.get("attachmentCount"),
            "attachmentIds": attachment_ids if attachment_ids else None,
            "assigneeEmail": assignee.get("email"),
            "assigneeName": " ".join(
                part for part in [assignee.get("firstName"), assignee.get("lastName")] if part
            ) or None,
        })
    return pd.DataFrame(rows)


try:
    sample_page = client.list_tickets(ORGANIZATION_UUID, limit=20)
    results = sample_page.get("results", [])
    print(f"✅ Generated DataFrame with {len(results)} tickets\n")
    
    # Display DataFrame
    df = tickets_to_dataframe(results)
    display(df)
except TicketApiError as exc:
    print(exc)

✅ Generated DataFrame with 20 tickets



,id,title,status,priority,type,createdAt,updatedAt,commentCount,attachmentCount,attachmentIds,assigneeEmail,assigneeName
0,17763977,[MEDIUM]Incident: Anomalous sign-in from a low...,PENDING,NORMAL,INCIDENT,2026-07-20T18:58:45Z,2026-07-20T18:58:47Z,1,1.0,None,kyle.hatlestad@arcticwolf.net,Kyle Hatlestad
1,17755066,[HIGH]Incident: Filename patterns associated w...,PENDING,HIGH,INCIDENT,2026-07-18T22:11:25Z,2026-07-18T22:11:37Z,1,NaN,None,kyle.hatlestad@arcticwolf.net,Kyle Hatlestad
2,17755035,[MEDIUM] Incident: Office 365: Evasive inbox r...,PENDING,NORMAL,INCIDENT,2026-07-18T21:45:41Z,2026-07-18T21:46:00Z,1,1.0,None,kyle.hatlestad@arcticwolf.net,Kyle Hatlestad
3,17755016,[MEDIUM]Incident: Member added to critical AD ...,PENDING,NORMAL,INCIDENT,2026-07-18T21:27:47Z,2026-07-18T21:27:51Z,1,1.0,None,kyle.hatlestad@arcticwolf.net,Kyle Hatlestad
4,17754907,[MEDIUM]Incident: Aurora Defense: PowerShell E...,PENDING,NORMAL,INCIDENT,2026-07-18T20:05:29Z,2026-07-18T20:05:38Z,1,1.0,None,kyle.hatlestad@arcticwolf.net,Kyle Hatlestad
5,17754796,[MEDIUM]Incident: Abnormal behavior: unusual a...,PENDING,NORMAL,INCIDENT,2026-07-18T18:23:30Z,2026-07-18T18:23:41Z,1,1.0,None,kyle.hatlestad@arcticwolf.net,Kyle Hatlestad
6,17754791,[MEDIUM]Incident: Mimikatz Lsadump Invocation ...,PENDING,NORMAL,INCIDENT,2026-07-18T18:11:49Z,2026-07-18T18:11:55Z,1,1.0,None,kyle.hatlestad@arcticwolf.net,Kyle Hatlestad
7,17754789,[MEDIUM]Incident: Mimikatz Initialization Dete...,PENDING,NORMAL,INCIDENT,2026-07-18T18:10:12Z,2026-07-18T18:10:24Z,1,1.0,None,kyle.hatlestad@arcticwolf.net,Kyle Hatlestad
8,17754769,[MEDIUM]Incident: File Create by WinRAR in Unc...,PENDING,NORMAL,INCIDENT,2026-07-18T17:55:26Z,2026-07-18T17:55:32Z,1,1.0,None,kyle.hatlestad@arcticwolf.net,Kyle Hatlestad
9,17754740,[HIGH]Incident: Backdoor.Win32.Qakbot.E (Initi...,PENDING,HIGH,INCIDENT,2026-07-18T17:39:33Z,2026-07-18T17:39:39Z,1,1.0,None,kyle.hatlestad@arcticwolf.net,Kyle Hatlestad


## Example 6 — Retrieve one ticket by ID

Set `TICKET_ID` to a real ticket ID from a previous list response.

Set `include_comments=True` when you need comments and attachment metadata in the response.

In [11]:
TICKET_ID = 17614272  # Replace with a real ticket ID.

try:
    ticket = client.get_ticket(
        ORGANIZATION_UUID,
        TICKET_ID,
        include_comments=True,
    )
    
    # Build markdown output
    markdown_output = f"""# Ticket #{ticket.get('id')} — {ticket.get('title')}

## Overview

| Field | Value |
|-------|-------|
| Status | {ticket.get('status')} |
| Priority | {ticket.get('priority')} |
| Type | {ticket.get('type')} |
| Created | {ticket.get('createdAt')} |
| Updated | {ticket.get('updatedAt')} |

## Details

**Organization UUID:** {ticket.get('organizationUuid')}

**Description:** {ticket.get('description', 'N/A')}

**Comments:** {ticket.get('commentCount', 0)}

**Attachments:** {ticket.get('attachmentCount', 0)}
"""
    
    # Add assignee if available
    assignee = ticket.get('assignee')
    if assignee:
        markdown_output += f"""
## Assignee

- **Name:** {assignee.get('firstName', '')} {assignee.get('lastName', '')}
- **Email:** {assignee.get('email')}
"""
    
    # Add comments if available
    comments = ticket.get('comments', [])
    if comments:
        markdown_output += f"""
## Comments ({len(comments)})

"""
        for comment in comments:
            author = comment.get('author', {})
            markdown_output += f"""### {author.get('firstName', '')} {author.get('lastName', '')} — {comment.get('createdAt')}

**Type:** {comment.get('type', 'N/A')}

{comment.get('body', '')}

---

"""
    
    # Add attachments if available
    attachments = ticket.get('attachments', [])
    if attachments:
        markdown_output += f"""
## Attachments ({len(attachments)})

| ID | Filename | Type | Created | Status |
|----|-----------|----|---------|--------|
"""
        for att in attachments:
            status = "🗑️ Deleted" if att.get('deleted') else "✅ Active"
            markdown_output += f"| {att.get('id')} | {att.get('filename', 'N/A')} | {att.get('contentType', 'N/A')} | {att.get('createdAt', 'N/A')} | {status} |\n"
    
    # Add full JSON
    markdown_output += f"""
## Full Ticket Object (JSON)

```json
{json.dumps(ticket, indent=2)}
```
"""
    
    print(markdown_output)
    
except TicketApiError as exc:
    print(exc)

# Ticket #17614272 — [HIGH]Incident: Filename patterns associated with SharpHound/BloodHound - Sample Co Llp

## Overview

| Field | Value |
|-------|-------|
| Status | CLOSED |
| Priority | HIGH |
| Type | INCIDENT |
| Created | 2026-06-27T22:11:53Z |
| Updated | 2026-07-07T23:02:39Z |

## Details

**Organization UUID:** None

**Description:** ## Summary

- BloodHound reconnaissance tool detected creating Active Directory enumeration files on desktop1 (10.171.170.101)
- Immediately isolate the affected host and investigate for unauthorized access

### What is it?

Arctic Wolf observed BloodHound/SharpHound execution on desktop1 at 2026-06-27T22:01:31Z UTC. The DecryptedSharpHound.exe process created multiple JSON files containing Active Directory reconnaissance data including groups, users, computers, and domain information.

This activity is considered malicious because BloodHound is a reconnaissance tool commonly used by threat actors to map Active Directory environments and identi

## Example 7 — Add a comment to a ticket

The request body requires `body`.

The comment body supports up to 65,535 characters.

In [ ]:
TICKET_ID = 12345  # Replace with a real ticket ID.
COMMENT_BODY = "API test comment from getting started notebook."

DRY_RUN = True

if DRY_RUN:
    print("DRY_RUN=True. No comment was added.")
    print({"ticket_id": TICKET_ID, "body": COMMENT_BODY})
else:
    try:
        added_comment = client.add_comment(
            ORGANIZATION_UUID,
            TICKET_ID,
            COMMENT_BODY,
        )
        print(added_comment)
    except TicketApiError as exc:
        print(exc)

## Example 8 — Close a ticket

Closing a ticket uses a `POST` request and accepts an optional `comment` field.

Keep `DRY_RUN=True` until you intentionally want to close a real ticket.

In [ ]:
TICKET_ID = 12345  # Replace with a real ticket ID.
CLOSE_COMMENT = "Closing via Ticket API after validation."

DRY_RUN = True

if DRY_RUN:
    print("DRY_RUN=True. No ticket was closed.")
    print({"ticket_id": TICKET_ID, "comment": CLOSE_COMMENT})
else:
    try:
        closed_ticket = client.close_ticket(
            ORGANIZATION_UUID,
            TICKET_ID,
            comment=CLOSE_COMMENT,
        )
        print(closed_ticket)
    except TicketApiError as exc:
        print(exc)

## Example 9 — Retrieve an attachment download URL

The attachment endpoint returns a pre-signed URL. The URL expires after a limited time.

Use attachment metadata from a ticket response where `includeComments=True`.

Do not request download URLs for attachments where `deleted=True`.

In [ ]:
import json

TICKET_ID = 17614272       # Replace with a real ticket ID.
ATTACHMENT_ID = 52265706791451   # Replace with a real attachment ID.

try:
    attachment_url_response = client.get_attachment_url(
        ORGANIZATION_UUID,
        TICKET_ID,
        ATTACHMENT_ID,
    )
    print(json.dumps(attachment_url_response, indent=2))
except TicketApiError as exc:
    print(exc)

## Example 10 — Download an attachment from the pre-signed URL

This separate request does not use the Ticket API bearer token. It uses the returned pre-signed URL.

Only run this when you trust the file type and destination path.

In [15]:
from pathlib import Path
from urllib.parse import urlparse


def download_presigned_url(url: str, destination_path: str | Path, allowed_domains: list[str] | None = None) -> Path:
    """Download from presigned URL with domain validation.
    
    Args:
        url: Presigned download URL
        destination_path: Local file path to save to
        allowed_domains: List of allowed domain names (e.g., ['*.arcticwolf.com', 'storage.api.com'])
                         If None, no domain validation is performed (not recommended)
    
    Returns:
        Path to downloaded file
    
    Raises:
        ValueError: If URL domain is not in allowed list
    """
    # Validate URL origin if allowed_domains is specified
    if allowed_domains:
        parsed = urlparse(url)
        hostname = parsed.hostname or ""
        is_allowed = any(
            domain.startswith("*.") and hostname.endswith(domain[1:]) or
            hostname == domain
            for domain in allowed_domains
        )
        if not is_allowed:
            raise ValueError(
                f"URL domain '{hostname}' not in allowed domains: {allowed_domains}"
            )
    else:
        print("⚠️ WARNING: Downloading from URL without domain validation. This is a security risk.")

    destination = Path(destination_path)
    with requests.get(url, stream=True, timeout=60) as response:
        response.raise_for_status()
        with destination.open("wb") as file_obj:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    file_obj.write(chunk)
    return destination


# Example usage with domain validation:
# response = client.get_attachment_url(ORGANIZATION_UUID, TICKET_ID, ATTACHMENT_ID)
# saved_path = download_presigned_url(
#     response["url"],
#     "attachment.bin",
#     allowed_domains=["*.arcticwolf.com", "s3.amazonaws.com"]  # Whitelist trusted domains
# )


## Common error responses

The API returns a JSON error envelope with at least `code`, and often `description`.

Common HTTP responses:

| Status | Meaning |
|---:|---|
| `400` | Invalid request. Example: `limit` greater than the allowed maximum. |
| `401` | Missing, invalid, or expired authentication token. |
| `403` | Authenticated but insufficient permissions. |
| `404` | Ticket or attachment was not found. |
| `500` | Internal server error. |

The `TicketApiError` class above extracts `code` and `description` when present.

## Minimal copy-paste script

This is a compact version for vibe-coded prototypes.

In [ ]:
import requests

# Reuses TOKEN, BASE_URL, and ORGANIZATION_UUID resolved in the Setup and
# Authentication cell above, instead of re-deriving them with a third copy of
# the base-URL resolution logic (the notebook previously had three separate
# implementations of this across two cells and ticket_api_client.py).
if not TOKEN or not ORGANIZATION_UUID:
    raise ValueError("Run the Setup and Authentication cell above first.")

headers = {
    "Authorization": f"Bearer {TOKEN}",
    "Accept": "application/json",
    "Content-Type": "application/json",
}

response = requests.get(
    f"{BASE_URL}/api/v1/organizations/{ORGANIZATION_UUID}/tickets",
    headers=headers,
    params={
        "status": "OPEN,PENDING",
        "priority": "HIGH,URGENT",
        "limit": 20,
        "offset": 0,
        "includeComments": "false",
    },
    timeout=30,
)
response.raise_for_status()

tickets = response.json()
tickets


## Safe extension points

Useful next steps for prototype code:

- Add retries for transient `429`, `500`, `502`, `503`, and `504` responses if those are observed in your environment.
- Add structured logging around method, path, status code, and elapsed time. Do not log tokens.
- Store tokens in a secret manager for deployed workflows.
- Add idempotency checks before commenting or closing tickets.
- Keep destructive operations behind an explicit `DRY_RUN` flag until tested.